# Comparativa de ASR usando SBERT (Cosine Similarity)

Este notebook calcula la similitud semántica entre las transcripciones normalizadas (`text_normalized`) y las oraciones de referencia (ground truth) utilizando **Sentence-BERT (SBERT)** y **similitud de coseno**.

A diferencia de BERTScore que trabaja a nivel de tokens, SBERT genera embeddings para toda la oración y compara estos vectores en un espacio semántico.

In [1]:
# Instalar dependencias si no están presentes
!pip install sentence-transformers pandas numpy torch

In [3]:
import pandas as pd
import json
from sentence_transformers import SentenceTransformer, util
import torch
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


## 1. Cargar Datos

In [5]:
# En Colab: Drive. En VS Code o local: rutas del proyecto.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = '/content/drive/MyDrive/'
    dataset_path = os.path.join(base_path, 'dataset_to_sbert.csv')
    ground_truth_path = os.path.join(base_path, 'ground_truth.json')
except ImportError:
    dataset_path = 'dataset_to_sbert.csv'
    ground_truth_path = 'ground_truth.json'
    print("Usando rutas locales (VS Code o ejecución local).")

# Cargar el dataset normalizado
df = pd.read_csv(dataset_path)

# Cargar el ground truth
with open(ground_truth_path, 'r') as f:
    ground_truth = json.load(f)

# Crear un diccionario para mapear id -> texto de referencia
ref_dict = {item['id']: item['text'] for item in ground_truth}

# Verificar las primeras filas
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,person,audio,noise,snr,provider,text,status,transcription_time,text_normalized
0,p7,1,cafe,0dB,custom,Genera una cotización para el cliente con fáci...,success,1.65,genera 1 cotización para el cliente con fácil ...
1,p7,1,cafe,5dB,custom,Genera una cotización para el cliente con PUC ...,success,1.56,genera 1 cotización para el cliente con puc fa...
2,p7,1,cafe,10dB,custom,Genera una cotización para el cliente con Puff...,success,1.59,genera 1 cotización para el cliente con puffaz...
3,p7,1,clean,clean,custom,Genera una cotización para el cliente con FooF...,success,1.66,genera 1 cotización para el cliente con foofac...
4,p7,1,traffic,0dB,custom,"genera una cotización para el cliente fácil, c...",success,1.53,genera 1 cotización para el cliente fácil con ...


## 2. Preparar Candidatos y Referencias

In [6]:
# Asegurarse de que la columna 'audio' sea string para hacer el mapeo con el id del json
df['audio'] = df['audio'].astype(str)

# Obtener la lista de candidatos (transcripciones normalizadas)
# Rellenar NaNs con string vacío por si acaso
cands = df['text_normalized'].fillna('').tolist()

# Obtener la lista de referencias correspondientes usando la columna 'audio' (que es el ID)
refs = [ref_dict.get(audio_id, "") for audio_id in df['audio']]

# Verificar que tienen la misma longitud
assert len(cands) == len(refs), "Error: La longitud de candidatos y referencias no coincide."

print(f"Total de pares a evaluar: {len(cands)}")
print(f"Ejemplo candidato: {cands[0]}")
print(f"Ejemplo referencia: {refs[0]}")

Total de pares a evaluar: 6000
Ejemplo candidato: genera 1 cotización para el cliente con fácil con 5 monitores led y 3 soportes de pared
Ejemplo referencia: genera 1 cotización para el cliente compufacil con 5 monitores led y 3 soportes de pared


## 3. Calcular SBERT Similarity (Cosine)

Utilizaremos un modelo multilingüe pre-entrenado de `sentence-transformers`.
El modelo `paraphrase-multilingual-mpnet-base-v2` es uno de los mejores para tareas de similitud semántica en varios idiomas.

In [12]:
# Cargar el modelo SBERT
# (Opcional) Silenciar warning de pesos no usados al cargar; es normal en estos modelos.
import logging
logging.getLogger('transformers').setLevel(logging.ERROR)

# Opción 1: Modelo multilingüe robusto (Recomendado, excelente rendimiento en español)
model_name = 'paraphrase-multilingual-mpnet-base-v2'

# Opción 2: Modelo específico para español (Entrenado solo en español)
# model_name = 'hiiamsid/sentence_similarity_spanish_es'

print(f"Cargando modelo SBERT: {model_name}...")
model = SentenceTransformer(model_name, device=device)

# Verificar que el modelo incluye la capa de pooling (no solo el transformer base)
print("Módulos del modelo (debe incluir Transformer + Pooling):", [m.__class__.__name__ for m in model])
print(f"Dimensión del embedding de oración: {model.get_sentence_embedding_dimension()}")

# Calcular embeddings para candidatos y referencias
print("Generando embeddings para candidatos...")
embeddings_cands = model.encode(cands, convert_to_tensor=True, show_progress_bar=True)

print("Generando embeddings para referencias...")
embeddings_refs = model.encode(refs, convert_to_tensor=True, show_progress_bar=True)

# Calcular similitud de coseno par a par
print("Calculando similitud de coseno...")
cosine_scores = torch.nn.functional.cosine_similarity(embeddings_cands, embeddings_refs)

# Convertir a numpy y guardar en el DataFrame
df['sbert_similarity'] = cosine_scores.cpu().numpy()

print("Cálculo completado.")

Cargando modelo SBERT: paraphrase-multilingual-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Módulos del modelo (debe incluir Transformer + Pooling): ['Transformer', 'Pooling']
Dimensión del embedding de oración: 768
Generando embeddings para candidatos...


Batches:   0%|          | 0/188 [00:00<?, ?it/s]

Generando embeddings para referencias...


Batches:   0%|          | 0/188 [00:00<?, ?it/s]

Calculando similitud de coseno...
Cálculo completado.


In [13]:
# Prueba rápida
test_frase = "prueba de dimensiones"
vector = model.encode(test_frase)

print(f"Forma del vector: {vector.shape}")

Forma del vector: (768,)


## 4. Analizar Resultados

In [8]:
# Mostrar estadísticas descriptivas de los scores
print("Estadísticas de SBERT Similarity:")
print(df['sbert_similarity'].describe())

# Mostrar promedio agrupado por proveedor (provider)
if 'provider' in df.columns:
    print("\nPromedio de SBERT Similarity por proveedor:")
    print(df.groupby('provider')['sbert_similarity'].mean().sort_values(ascending=False))

# Mostrar promedio agrupado por nivel de ruido (noise)
if 'noise' in df.columns:
    print("\nPromedio de SBERT Similarity por tipo de ruido:")
    print(df.groupby('noise')['sbert_similarity'].mean().sort_values(ascending=False))

# Mostrar promedio agrupado por SNR
if 'snr' in df.columns:
    print("\nPromedio de SBERT Similarity por nivel de ruido (SNR):")
    print(df.groupby('snr')['sbert_similarity'].mean().sort_values(ascending=False))

Estadísticas de SBERT Similarity:
count    6000.000000
mean        0.963549
std         0.080728
min         0.083842
25%         0.966107
50%         1.000000
75%         1.000000
max         1.000000
Name: sbert_similarity, dtype: float64

Promedio de SBERT Similarity por proveedor:
provider
google    0.973678
custom    0.965570
amazon    0.959942
azure     0.955006
Name: sbert_similarity, dtype: float32

Promedio de SBERT Similarity por tipo de ruido:
noise
clean        0.987967
cafe         0.968169
traffic      0.965665
warehouse    0.948673
Name: sbert_similarity, dtype: float32

Promedio de SBERT Similarity por nivel de ruido (SNR):
snr
clean    0.987967
10dB     0.981184
5dB      0.973259
0dB      0.928064
Name: sbert_similarity, dtype: float32


In [14]:
# Guardar el dataframe con los scores
output_filename = 'sbert_dataset.csv'
df.to_csv(output_filename, index=False)
print(f"Resultados guardados en {output_filename}")

# Si estamos en Colab, guardar también en Drive
try:
    from google.colab import drive
    drive_path = os.path.join(base_path, output_filename)
    df.to_csv(drive_path, index=False)
    print(f"Resultados guardados en Drive: {drive_path}")
except ImportError:
    pass

Resultados guardados en sbert_dataset.csv
Resultados guardados en Drive: /content/drive/MyDrive/sbert_dataset.csv


In [16]:
# Mostrar los 20 registros con menor similitud
peores_20 = df.nsmallest(20, 'sbert_similarity')
display(peores_20[['audio', 'provider','snr', 'text_normalized', 'sbert_similarity']])

,audio,provider,snr,text_normalized,sbert_similarity
3637,4,google,0dB,quiero 1 persona para mi mejor,0.083842
2437,4,custom,0dB,que le dieron al paciente para vivir en corte ...,0.125550
2447,5,custom,0dB,vista la última postura del presidente de la p...,0.135331
4797,15,azure,0dB,hundido durante no entiendo primero,0.145958
2247,15,custom,0dB,gracias por ver el video,0.227678
2440,5,custom,0dB,lucha la última partida del cliente velasco y ...,0.321927
3647,5,google,0dB,busca lo último en cultura y entretenimiento d...,0.347847
1440,10,custom,0dB,de la marca dura vista,0.370478
5434,4,google,0dB,mira 1 persona para mi laptop,0.381421
30,4,custom,0dB,tenemos 1 reforma para el importe incluye 2 la...,0.388768
